In [ ]:
import pandas as pd
import numpy as np
import sys
from tqdm.auto import tqdm
from antropy import perm_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, lowpass_filter_causal, bootstrap_report

In [ ]:
sampling_rate = 100 # Hz, sensor sampling frequency
lpf_cutoff = 10 # Hz, FIR lowpass cutoff
lpf_order = 64 # FIR filter taps
pe_order = 5 # permutation entropy embedding dimension
pe_delay = int(sampling_rate / (2 * lpf_cutoff)) # PE delay (Nyquist-matched)
pe_w = 1000 # entropy window size in samples
pe_s = 100 # window step size in samples
alpha = 0.023 # false-alarm bound
glue = False # set True to glue metrics for result collection
per_ride_path = None # set a path to save per-ride outcomes as CSV

In [ ]:
data = pd.read_parquet("../data/data.parquet")
data

In [ ]:
labels = data.groupby(["ride_id", "participant_id"])["run"].first().reset_index()
labels

In [ ]:
features = ["ax", "ay", "az", "rx", "ry", "rz", "throttle"]

data_np = reshape_to_numpy(
    data,
    features = features,
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_lpf = lowpass_filter_causal(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=lpf_cutoff,
    gyro_cutoff=lpf_cutoff,
    order=lpf_order,
    fs=sampling_rate,
)

In [ ]:
# Precompute windowed entropies for all rides.
n_channels = len(features)
all_windowed_ent = []

for ride in tqdm(data_np_lpf, desc="windowed entropy"):
    # get row indices without any NaN
    valid = np.flatnonzero(~np.any(np.isnan(ride), axis=1))
    # strip leading (FIR warm-up) and trailing NaNs
    ride_trimmed = ride[valid[0]:valid[-1] + 1]
    # sliding-window permutation entropy per channel
    n_steps = max(0, (len(ride_trimmed) - pe_w) // pe_s + 1)
    ride_ent = np.empty((n_steps, n_channels))
    for t in range(n_steps):
        window = ride_trimmed[t * pe_s : t * pe_s + pe_w]
        for j in range(n_channels):
            ride_ent[t, j] = perm_entropy(window[:, j],
                                          order=pe_order, delay=pe_delay, normalize=True)
    all_windowed_ent.append(ride_ent)

In [ ]:
# Within-subject centering (WSC): subtract per-channel mean of the
# participant's other rides to remove between-subject variability
# without needing to know which ride is sober.
pids = labels["participant_id"].values
all_windowed_wsc = []

for i in range(len(labels)): # i is a positional index, not a label
    # indices of the same participant's other rides
    same_pid = np.flatnonzero(pids == pids[i])
    other_idx = same_pid[same_pid != i]
    # collect their entropy windows
    other_windows = [all_windowed_ent[j] for j in other_idx]
    # per-channel mean across those rides
    mu = np.nanmean(np.vstack(other_windows), axis=0)
    # subtract to center within subject
    all_windowed_wsc.append(all_windowed_ent[i] - mu)

In [ ]:
# LOPO CV
logo = LeaveOneGroupOut()
results = []

for train_idx, test_idx in tqdm(list(logo.split(labels, labels["run"], labels["participant_id"])), desc="LOPO"):
    # split training windows by label
    train_labels = labels.iloc[train_idx].reset_index(drop=True)
    tr_sober = [all_windowed_wsc[train_idx[j]] for j in range(len(train_labels)) if train_labels["run"].iloc[j] == "Sober"]
    tr_imp   = [all_windowed_wsc[train_idx[j]] for j in range(len(train_labels)) if train_labels["run"].iloc[j] != "Sober"]

    # fit a single logistic regression; P(impaired | x) as detector score
    X_tr = np.vstack(tr_sober + tr_imp)
    y_tr = np.concatenate([np.zeros(sum(map(len, tr_sober))), np.ones(sum(map(len, tr_imp)))])
    clf = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)

    # compute per-window LR score path for held-out participant
    for ride_idx in test_idx:
        # load test ride windows
        ride_wsc = all_windowed_wsc[ride_idx]

        # raw LR score per window (no averaging)
        score_path = clf.predict_proba(ride_wsc)[:, 1].tolist()

        results.append({
            "run": labels["run"].iloc[ride_idx],
            "participant_id": labels["participant_id"].iloc[ride_idx],
            "score_path": score_path,
        })

df_res = pd.DataFrame(results)
df_res

In [ ]:
tau = 1 - alpha # naive fixed threshold (matches batch LR; no per-ride calibration)
detection_step = [] # detection steps

# detection: first window where the LR score > tau
for path in df_res["score_path"]:
    alarm_steps = np.flatnonzero(np.asarray(path) > tau)
    detection_step.append(int(alarm_steps[0]) if len(alarm_steps) else None)

# Group results in a DF
df_res["detection_step"] = detection_step
df_res["detected"] = [s is not None for s in detection_step]
df_res["detection_time_s"] = [
    (s * pe_s + pe_w) / sampling_rate if s is not None else None for s in detection_step
]
df_res

In [ ]:
_ = bootstrap_report(df_res, glue=glue, per_ride_path=per_ride_path)